## Dependencies

In [1]:
# Install dependencies
%pip install langchain langchain-core langchain-openai langchain-community langchain-text-splitters chromadb pypdf python-dotenv unstructured beautifulsoup4 lxml python-docx pypandoc rank-bm25 -q

# Verify installation
import sys
print(f"Python version: {sys.version}")

# Verify key packages are installed
try:
    import langchain
    import langchain_core
    print(f"LangChain version: {langchain.__version__}")
    print("All core packages installed successfully!")
except ImportError as e:
    print(f"⚠️  Error: {e}")
    print("Please ensure all packages are installed correctly.")


Note: you may need to restart the kernel to use updated packages.
Python version: 3.13.12 (v3.13.12:1cbe4818347, Feb  3 2026, 13:36:53) [Clang 16.0.0 (clang-1600.0.26.6)]
LangChain version: 1.2.12
All core packages installed successfully!


## Setup & Imports

In [2]:
# Standard library imports
import os
from pathlib import Path
from dotenv import load_dotenv

# LangChain core imports
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader, UnstructuredODTLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# LangChain retrieval imports
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# Website scraping strainer
from bs4 import SoupStrainer

#download pypandoc
import pypandoc
pypandoc.download_pandoc()

# Load environment variables - check current directory first, then parent
current_dir = Path.cwd()
env_path = current_dir / '.env'
if env_path.exists():
    load_dotenv(env_path)
    print(f"Loading .env from: {env_path}")
else:
    # Try parent directory
    parent_env = current_dir.parent / '.env'
    if parent_env.exists():
        load_dotenv(parent_env)
        print(f"Loading .env from: {parent_env}")
    else:
        # Fallback to default behavior (searches current and parent directories)
        load_dotenv()
        print("⚠️  No .env file found. Using default load_dotenv() search")

# Verify API key is set
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("⚠️  WARNING: OPENAI_API_KEY not found in environment variables!")
    print("Please create a .env file in this directory with your OpenAI API key:")
    print(f"   {current_dir / '.env'}")
    print("\nFormat: OPENAI_API_KEY=sk-...")
else:
    # Validate API key format (basic check)
    if not api_key.startswith('sk-'):
        print("⚠️  WARNING: API key format looks incorrect. Should start with 'sk-'")
    else:
        print("OpenAI API key loaded successfully!")
        # Test the API key by making a simple call
        try:
            test_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, max_tokens=5)
            test_llm.invoke("Hi")
            print("API key validated successfully!")
        except Exception as e:
            print(f"❌ ERROR: API key validation failed!")
            print(f"   Error: {str(e)}")
            print("\n💡 Please check:")
            print("   1. Your API key is correct (get it from https://platform.openai.com/api-keys)")
            print("   2. You have sufficient credits in your OpenAI account")
            print("   3. The .env file is in the correct location")
            raise

print("\nAll imports successful!")
print("Using RetrievalQA with a custom prompt template")

USER_AGENT environment variable not set, consider setting it to identify your requests.
x .
x ./usr
x ./usr/local
x ./usr/local/bin
x ./usr/local/bin/pandoc
x ./usr/local/bin/._pandoc
x ./usr/local/bin/pandoc-lua
x ./usr/local/bin/._pandoc-lua
x ./usr/local/bin/pandoc-server
x ./usr/local/bin/._pandoc-server
x ./usr/local/._bin
x ./usr/local/share
x ./usr/local/share/man
x ./usr/local/share/man/man1
x ./usr/local/share/man/man1/pandoc-lua.1
x ./usr/local/share/man/man1/._pandoc-lua.1
x ./usr/local/share/man/man1/pandoc-server.1
x ./usr/local/share/man/man1/._pandoc-server.1
x ./usr/local/share/man/man1/pandoc.1
x ./usr/local/share/man/man1/._pandoc.1
x ./usr/local/share/man/._man1
x ./usr/local/share/._man
x ./usr/local/._share
x ./usr/._local
x ./._usr


Loading .env from: /Users/kaliml/Cursor Projects/rag-qa-system/.env
OpenAI API key loaded successfully!
API key validated successfully!

All imports successful!
Using RetrievalQA with a custom prompt template


## Step 1: Load
- Load documents from three data types, and then preview the start and end of each document/page:
    - Live web page
    - PDF
    - ODT

In [3]:

# Load website
web_loader = WebBaseLoader(
    "https://brokenscience.org/what-is-fitness-crossfit/",
    bs_kwargs={
        "parse_only": SoupStrainer(
            class_=["post-content"]  # only pull article content, not website navigation, etc.
        )
    }
)

#Load PDF
pdf_loader = PyPDFLoader("docs/safety_and_intensity_crossfit.pdf")

# Load ODT
odt_loader = UnstructuredODTLoader("docs/The_Foundation_Of_CrossFit_ Moving_With_Proper_Mechanics.odt")

# Combine all documents into one list
documents = (
    web_loader.load() +
    pdf_loader.load() +
    odt_loader.load()
    
)

# Trim the bottom noise from the webpage
cutoff_phrase = "This article, by BSI’s co-founder"
idx = documents[0].page_content.find(cutoff_phrase)
if idx != -1:
    documents[0].page_content = documents[0].page_content[:idx]

#Display document previews for review
print("-" * 60)
print(f"Loaded {len(documents)} document(s)")

for index, document in enumerate(documents):
    print(f"Document #", index+1)
    print(f"Verify start of document looks clean:")
    print("-" * 3)
    print(document.page_content[:300])
    print("-" * 3)
    print(f"\nVerify end of document looks clean:")
    print("-" * 3)
    print(document.page_content[-300:])
    print("-" * 3)
    print(f"\nDocument metadata: {document.metadata}")
    print("-" * 80)





------------------------------------------------------------
Loaded 5 document(s)
Document # 1
Verify start of document looks clean:
---

What Is Fitness and Who Is Fit?
In 1997, Outside Magazine crowned triathlete Mark Allen “the fittest man on Earth.” Let us just assume for a moment that this famous six-time winner of the IronMan Triathlon is the fittest of the fit. Then what title do we bestow on the decathlete Simon Poelman, who 
---

Verify end of document looks clean:
---
neffective. The need for specificity is nearly completely met by regular practice and training within the sport, not in the strength-and-conditioning environment. Our terrorist hunters, skiers, mountain bikers and housewives have found their best fitness from the same regimen.

Cover image: Dave Re

---

Document metadata: {'source': 'https://brokenscience.org/what-is-fitness-crossfit/'}
--------------------------------------------------------------------------------
Document # 2
Verify start of document looks c

## Step 2: Chunk

### 2.1 Create RecursiveCharacterTextSplitter
- Split text into manageable segments based on characters and predefined criteria

In [4]:
# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # Size of each chunk
    chunk_overlap=100,   # Overlap between chunks (important!)
    separators=["\n\n", "\n", ". ", " ", ""]  # Try to split at these boundaries
)

# Split documents into chunks
chunks = text_splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks")
print(f"\nChunk statistics:")
print(f"  Average chunk size: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} characters")
print(f"  Min chunk size: {min(len(c.page_content) for c in chunks)} characters")
print(f"  Max chunk size: {max(len(c.page_content) for c in chunks)} characters")

Split into 87 chunks

Chunk statistics:
  Average chunk size: 542 characters
  Min chunk size: 5 characters
  Max chunk size: 800 characters


### 2.2 Experiment with Chunk Sizes
- Determine whether a smaller or larger chunk size works best for this use case.  After experimentation size 800 was chosen to preserve complete thoughts across multi-sentence answers.  

In [5]:
# Chunk size experiment — uses separate variables so the main `chunks` pipeline is unaffected
for size in [500, 800]:
    trial = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=100).split_documents(documents)
    sizes = [len(c.page_content) for c in trial]
    print(f"chunk_size={size}: {len(trial)} chunks | avg={sum(sizes)//len(sizes)} | min={min(sizes)} | max={max(sizes)}")

chunk_size=500: 137 chunks | avg=367 | min=5 | max=499
chunk_size=800: 87 chunks | avg=546 | min=5 | max=800


## Step 3: Embed + Store

- Embed chunks using OpenAI Embeddings, then store in a ChromaDB vector database


In [6]:
import shutil

# Clear the vector store between runs (this allows for changing chunking settings and testing fresh)
try:
    del vectorstore
except NameError:
    pass  # No existing vectorstore, that's fine

# Now clear the folder
db_path = "./rag_vector_db"
if os.path.exists(db_path):
    shutil.rmtree(db_path)
    print("Cleared existing vector database")

# Create fresh
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
)
print("Created fresh vector database!")
print(f"Stored {len(chunks)} document chunks")

# Create retrievers + ensemble (BM25 + vector similarity)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
bm25_retriever = BM25Retriever.from_documents(chunks, k=5)
ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.4, 0.6],
)

print("Created ensemble retriever (vector + BM25)")


Created fresh vector database!
Stored 87 document chunks
Created ensemble retriever (vector + BM25)


# Step 4: Test Retrieval (before wiring up the LLM!)


In [7]:
#Retrieval Test Question setup
questions = [
    "What is a hopper used for?",
    "What is the risk of squatting with a rounded back?",
    "What is the coach responsible for?",
]

for question in questions:
    results = ensemble_retriever.invoke(question)[:3]

    print("-" * 80)
    print(f"\nTEST QUERY: '{question}'")
    print(f"\nRETRIEVED {len(results)} RELEVANT CHUNKS:")
    print("-" * 40)
    for i, doc in enumerate(results, 1):
        print(f"\n{i}. {doc.page_content[:200]}...")
        print(f"   METADATA: {doc.metadata}")
        print("-" * 15)


--------------------------------------------------------------------------------

TEST QUERY: 'What is a hopper used for?'

RETRIEVED 3 RELEVANT CHUNKS:
----------------------------------------

1. At 15 pull-ups and dips each, it is time to start working regularly on a muscle-up. The muscle-up is moving from a hanging position below the rings to a supported position, arms extended, above the ri...
   METADATA: {'source': 'https://brokenscience.org/what-is-fitness-crossfit/'}
---------------

2. (Ed. – Thanks to Jim Crawley and Bruce Evans of Dynamax)
CrossFit’s Second Fitness Standard
The essence of this model is the view that fitness is about performing well at any and every task imaginable...
   METADATA: {'source': 'https://brokenscience.org/what-is-fitness-crossfit/'}
---------------

3. What Is Fitness and Who Is Fit?
In 1997, Outside Magazine crowned triathlete Mark Allen “the fittest man on Earth.” Let us just assume for a moment that this famous six-time winner of the IronMan 

### 4.2 Chunk Assessment

NOTE:  These results vary slightly every time the data is re-run.  

**Q1: "What is a hopper used for?"**
(**Tests a specific keyword**)
- Retrieved Chunk 1: ❌ Not relevant — discusses muscle-ups, unrelated to hoppers
- Retrieved Chunk 2: ✅ Relevant - directly describes how a hopper is used
- Retrieved Chunk 3: ✅ Relevant - this is the chunk directly preceeding the hopper chunk 
- Note: The hopper concept (randomized workout selection) is in the documents but the keyword "hopper" was buried inside a larger topic.  It was missed by the similarity search (which runs first), but then picked up by BM25 in Chunk 2. 

**Q2: "What is the risk of squatting with a rounded back?"**
(**Tests a specific scenario with focused mentions**)
- Retrieved Chunk 1: ✅ Relevant — directly mentions that a squat with a rounded back puts excessive strain on the spine, leading to potential disc herniation or muscle strains
- Retrieved Chunk 2: ✅ Relevant — discusses proper form
- Retrieved Chunk 3: ❌ Not relevant — intro paragraph about fitness standards ("What Is Fitness and Who Is Fit?"); no connection to squatting
- Note:  It found the correct information in Chunk 1.  This was the test of isolated reference, so we should expect this result.  There are not 3 chunks containing this information.

**Q3: "What is the coach responsible for?"**
(**Tests a broader concept with data spread across multiple chunks**)
- Retrieved Chunk 1: ✅ Relevant — directly describes coaches guiding athletes through mechanics, consistency, and intensity progression before increasing loads
- Retrieved Chunk 2: ✅ Relevant — covers the coach's responsibility for warm-ups, athlete readiness assessment, cool-downs, and load management
- Retrieved Chunk 3: ✅ Relevant — describes the coach's role in threshold training and balancing intensity with sound mechanics for every athlete

## Step 5: Build the RAG Chain

- Wire up RetrievalQA with a custom prompt template
- Run the same 3 queries through the full chain
- Print the generated answers

### 5.1:  Set up the RetrievalQA Chain

In [8]:
# Initialize LLM
model = ChatOpenAI(
    model="gpt-4o-mini",  # Fast and cost-effective
    temperature=0  # Deterministic responses
)

# Custom prompt template for RetrievalQA
qa_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "You are a helpful assistant for document-grounded QA.\n"
        "Use only the provided context to answer the question.\n"
        "If the context does not contain enough information, say so clearly.\n"
        "Do not guess and do not use outside knowledge.\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}\n"
        "Answer:"
    ),
)

# Ensure retriever exists if Step 3 wasn't run in this session
if "ensemble_retriever" not in globals():
    if "vectorstore" in globals() and "chunks" in globals():
        print("ensemble_retriever not found; rebuilding from vectorstore + chunks...")
        vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
        bm25_retriever = BM25Retriever.from_documents(chunks, k=5)
        ensemble_retriever = EnsembleRetriever(
            retrievers=[vector_retriever, bm25_retriever],
            weights=[0.4, 0.6],
        )
    else:
        raise RuntimeError(
            "Missing retrieval prerequisites. Run Step 3 (Embed + Store) first, "
            "or initialize `vectorstore` and `chunks` before Step 5.1."
        )

# Build RetrievalQA chain with the existing ensemble retriever
qa_chain = RetrievalQA.from_chain_type(
    llm=model,
    chain_type="stuff",
    retriever=ensemble_retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": qa_prompt},
)

print("\nRETRIEVALQA CONFIGURATION:")
print(f"  LLM: {model.model_name}")
print("  CHAIN: RetrievalQA (stuff)")
print("  RETRIEVER: EnsembleRetriever (top 3 shown)")


RETRIEVALQA CONFIGURATION:
  LLM: gpt-4o-mini
  CHAIN: RetrievalQA (stuff)
  RETRIEVER: EnsembleRetriever (top 3 shown)


### 5.2 Test the RAG Chain output

In [9]:
# Use the questions defined previously

for question in questions:
    print(f"\n{'='*70}")
    print(f"❓ Question: {question}")
    print(f"{'='*70}\n")

    result = qa_chain.invoke({"query": question})
    answer = result["result"]
    source_docs = result.get("source_documents", [])[:3]

    print(f"💬 Answer:\n{answer}\n")
    print("📚 Sources:")
    for i, doc in enumerate(source_docs, 1):
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", doc.metadata.get("page_label", "n/a"))
        snippet = doc.page_content[:220].replace("\n", " ")
        print(f"  {i}. source={source} | page={page}")
        print(f"     snippet: {snippet}...")

    print("-" * 70)


❓ Question: What is a hopper used for?

💬 Answer:
A hopper is used to represent a model of fitness where individuals are asked to perform feats randomly drawn from an infinite number of physical challenges. It illustrates the idea that fitness is about performing well at any and every task imaginable, without a selective mechanism in place.

📚 Sources:
  1. source=https://brokenscience.org/what-is-fitness-crossfit/ | page=n/a
     snippet: At 15 pull-ups and dips each, it is time to start working regularly on a muscle-up. The muscle-up is moving from a hanging position below the rings to a supported position, arms extended, above the rings. It is a combina...
  2. source=https://brokenscience.org/what-is-fitness-crossfit/ | page=n/a
     snippet: (Ed. – Thanks to Jim Crawley and Bruce Evans of Dynamax) CrossFit’s Second Fitness Standard The essence of this model is the view that fitness is about performing well at any and every task imaginable. Picture a hopper l...
  3. source=https:

## Step 6: Evaluate

### 6.1 Eval Setup
Questions were chosen to provide a mix of simple and complex answers.  A sixth question was added to target the BM25 algorithm with a very specific key word that only appeared one time across the documents.  

In [11]:
eval_questions = [
    "How do I achieve sustainable intensity?",
    "Why are warm-up routines important?",
    "What needs to happen before increasing load?",
    "Why would an athlete's intensity need to change?",
    "Who is responsible for managing threshold training?",
    "What is considered a pathological blood pressure?",
    "How much water should an athlete drink in a given 24 hour period?"
]

# Expected answers (for manual scoring)
expected_answers = [
    "Safety and proper mechanics are the foundation for Sustainable Intensity",
    "Warm-ups prepare the body for upcoming demands, allow coaches time for skill refinement and load management, and lay the groundwork for effective scaling. ... A key component of a session is a time to check in or assess your athlete’s readiness ... This can be achieved ... during the general and specific warm-up.",
    "Coaches guide athletes on a path to develop correct technique. After mechanics and consistency are established, coaches guide them deliberately through the charter of mechanics, consistency, intensity, plus threshold training.",
    "Relative intensity may change based on poor night's sleep or being in a period of high stress, in order to ensure proper body mechanics,  can be relative to the individual’s physical and psychological tolerance.",
    "The coach is responsible for managing this for every workout, for every athlete.",
    "A blood pressure of 160/95 is pathological.",
    "This is outside the scope of the data and should not receive an answer."
]

results = []

for i, question in enumerate(eval_questions):
    print(f"\n{'='*70}")
    print(f"❓ Question {i+1}: {question}")
    print(f"{'='*70}\n")

    result = qa_chain.invoke({"query": question})
    final_answer = result["result"]

    print(f"💬 Answer:\n{final_answer}\n")
    print(f"📋 Expected:\n{expected_answers[i]}\n")




❓ Question 1: How do I achieve sustainable intensity?

💬 Answer:
To achieve sustainable intensity in CrossFit, focus on the following key components:

1. **Proper Mechanics**: Master foundational movement patterns to minimize the risk of injury and improve performance. Good technique is essential for long-term success.

2. **Listen to Your Body**: Pay attention to how you feel during workouts. If something feels off or causes discomfort, stop and reassess your form.

3. **Progressive Intensity**: Gradually increase intensity over time while ensuring sound mechanics. This helps to push your boundaries safely.

4. **Effective Scaling**: Adjust workouts to meet your individual physical and psychological tolerances. This ensures that the intensity is appropriate for your current state.

5. **Balance of Intensity and Mechanics**: Work with a coach to find the right balance between pushing intensity and maintaining proper movement mechanics.

6. **Mindset Shift**: Understand that safety is 

### 6.2 Manual Scoring

Scoring Rubric:

- Retrieved = Did it retrieve the right chunks?
- Grounded = Is the answer grounded in the context (not hallucinated)?
- Correct = Is the answer actually correct?

----------------------------------------------

| Q  | Retrieved | Grounded | Correct |
|----|---------- |----------|---------|
| 1  | ✅        | ✅        | ✅      |
| 2  | ✅        | ✅        | ✅      |
| 3  | ✅        | ✅        | ✅      |
| 4  | ✅        | ✅        | ✅      |
| 5  | ✅        | ✅        | ✅      |
| 6  | ✅        | ✅        | ✅      |
| 7  | ✅        | ✅        | ✅      |

----------------------------------------------

Retrieval:  7/7 = 100%
Grounded/Faithfulness:   7/7 = 100%
Correctness:    7/7 = 100% 

----------------------------------------------

Factors that brought these scores into an accurate range:
- Adjusted model prompt for specificity and to only use data from the documents, not its own knowledge
- Implemented EnsembleRetriever to add a BM25 keyword lookup algorithm along with the similarity search.  
